# GPGPU Tasks 2026 — Colab Runner (CUDA)

**Runtime → Change runtime type → T4 GPU.**

Workflow: правишь код локально → `git push` в ветку `task02` → здесь ячейка 4 (pull + make + запуск).


In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. [1 раз за сессию] Клонирование + зависимости + первая сборка
BRANCH = "task02"
%cd /content
!rm -rf /content/GPGPUTasks2026
!git clone -b $BRANCH https://github.com/BOGOMOLOV-ARSENIQ/GPGPUTasks2026.git /content/GPGPUTasks2026
!chmod +x /content/GPGPUTasks2026/scripts/linux/*.sh
!/content/GPGPUTasks2026/scripts/linux/install_vulkan_sdk.sh --no-validation-layers
!sudo apt-get update -qq && sudo apt-get install -yqq libgtest-dev libgraphicsmagick++1-dev

In [ ]:
# 3. [1 раз за сессию] Конфигурация CMake (75 = T4)
!mkdir -p /content/GPGPUTasks2026/build
%cd /content/GPGPUTasks2026/build
!cmake -DCMAKE_BUILD_TYPE=Release -DGPU_CUDA_SUPPORT=ON -DCMAKE_CUDA_ARCHITECTURES=75 ..

In [ ]:
# 4. [Каждый раз после git push] Подтянуть код, пересобрать, запустить
%cd /content/GPGPUTasks2026
!git fetch -q origin && git reset -q --hard origin/$BRANCH && git log --oneline -1
%cd /content/GPGPUTasks2026/build
!make -j$(nproc) 2>&1 | tail -20

print("\n\033[94m=== libgpu_test ===\033[0m")
!AVK_ENABLE_VALIDATION_LAYERS=false ./libs/gpu/libgpu_test --gtest_filter=-vulkan.*
print("\n\033[94m=== main_aplusb ===\033[0m")
!./main_aplusb 0
print("\n\033[94m=== main_mandelbrot ===\033[0m")
!./main_mandelbrot 0
print("\n\033[94m=== main_sum ===\033[0m")
!./main_sum 0

In [ ]:
# 5. (опционально) Показать сохранённые картинки фрактала
from PIL import Image
from IPython.display import display
import glob
for p in sorted(glob.glob("/content/GPGPUTasks2026/build/*.bmp")):
    print(p); display(Image.open(p))

In [ ]:
# 6. По окончании — освободить GPU (экономит лимит)
from google.colab import runtime
runtime.unassign()
